# Practice 2.2 — Preprocessing & Data Preparation

This notebook covers **Phase 1–6** of the practice-2.2 pipeline: problem definition, data inventory, integrity verification (MD5), stratified group-aware splitting, exploratory data analysis, the preprocessing/augmentation transforms, and export of split indices for the training stage. Model training is intentionally **out of scope** for this notebook.

| Aspect | Value |
|---|---|
| Task | Supervised single-label image classification |
| Input | 224×224 cosmetic product images (3-channel RGB) |
| Classes | 10 (`body_wash`, `face_mask`, `facial_cleanser`, `lipstick`, `moisturizer`, `perfume`, `serum`, `shampoo`, `sunscreen`, `toner`) |
| Backbone (next stage) | ImageNet-pretrained ResNet18 |
| Split | 70% Train / 15% Validation / 15% Test (stratified, group-aware) |
| Data status | Pre-cleaned (no leaks, no duplicates, no blurry images) |
| Augmentation policy | Augment **only** the training split |


In [ ]:
import json, random, time
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image
import hashlib
from IPython.display import display
import sys
PROJECT_ROOT = Path("d:/Hoc_tap/TOTAL_LABPARACTICE_FOR_DEEP_LEARNING")
DATASET_ROOT = PROJECT_ROOT / "total_practice" / "practice_2_2" / "data_clean_224"
OUTPUT_DIR = PROJECT_ROOT / "total_practice" / "practice_2_2" / "outputs" / "practice_2_2"
REPORTS_DIR = PROJECT_ROOT / "total_practice" / "practice_2_2" / "reports" / "practice_2_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
IMAGE_SIZE = 224
BATCH_SIZE = 32
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
env_df = pd.DataFrame([
    ["Python",     f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"],
    ["PyTorch",    torch.__version__],
    ["CUDA avail", torch.cuda.is_available()],
    ["Device",     str(DEVICE)],
    ["Dataset",    DATASET_ROOT.name],
    ["Seed",       SEED],
    ["Image size", IMAGE_SIZE],
    ["Batch size", BATCH_SIZE],
], columns=["key", "value"])
display(env_df)


## Phase 3 — Data Inventory & Integrity Verification

Before any modeling, we re-verify the integrity of the cleaned dataset:

1. **No pre-generated augmentations.** Filenames follow `class_folder/<stem>.<ext>` with no `_augNNN` suffix. Every file in `data_clean_224` is a clean original.
2. **No cross-class leaks.** The same MD5 hash must not appear under two different class folders.
3. **No intra-class duplicates.** Two byte-identical files inside the same class are still a duplication that biases the split and inflates metrics.

We compute the MD5 of every file and group by hash. The number of unique hashes must equal the number of files, and each hash must map to a single class label.

In [ ]:
assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"

raw_dataset = datasets.ImageFolder(DATASET_ROOT, transform=None)
CLASS_NAMES = raw_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

h2f = defaultdict(list)
file_records = []
t0 = time.time()
for idx, (path, label) in enumerate(raw_dataset.samples):
    p = Path(path)
    with open(p, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()
    h2f[h].append((CLASS_NAMES[label], p.name))
    file_records.append({
        "index": idx,
        "path": str(p),
        "label": int(label),
        "class_name": CLASS_NAMES[label],
        "filename": p.name,
        "hash": h,
    })
md5_elapsed = time.time() - t0

df = pd.DataFrame(file_records)
total_files = len(df)
unique_hashes = df["hash"].nunique()
cross_class_leaks = sum(1 for h, lst in h2f.items() if len(set(c for c, _ in lst)) > 1)
intra_dups = sum(1 for h, lst in h2f.items() if len(lst) > 1)

integrity = pd.DataFrame([
    ["Total files",          total_files],
    ["Unique MD5 hashes",    unique_hashes],
    ["Cross-class leaks",    cross_class_leaks],
    ["Intra-class duplicates", intra_dups],
    ["Classes",              NUM_CLASSES],
    ["MD5 scan time (s)",    round(md5_elapsed, 2)],
], columns=["metric", "value"])
display(integrity)

assert cross_class_leaks == 0, "CRITICAL: cross-class leaks detected!"
assert intra_dups == 0, "WARNING: intra-class duplicates exist"
assert total_files == unique_hashes, "WARNING: some files are byte-identical"

per_class_counts = df["class_name"].value_counts().reindex(CLASS_NAMES)
display(per_class_counts.to_frame("count"))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=per_class_counts.index, y=per_class_counts.values, ax=ax, palette="viridis")
for i, v in enumerate(per_class_counts.values):
    ax.text(i, v + 1, str(v), ha="center", va="bottom", fontsize=9)
ax.set_title(f"Per-class image count (total={total_files})")
ax.set_xlabel("class"); ax.set_ylabel("# images")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "class_distribution_raw.png", dpi=120, bbox_inches="tight")
plt.show()

## Phase 5 — Stratified Group-Aware Split

Because no `_augNNN` files are present, no per-original augmentation grouping is required *today*. However, we still define the grouping rule as **`{class_id:02d}/{filename_stem}`** so the same code is correct if augmentations are introduced later — every augmented variant (`img_aug001.jpg`, `img_aug002.jpg`, …) of an original `img.jpg` shares the same stem and therefore the same group.

Splitting strategy:

- One group per unique `(class_id, filename_stem)`.
- For each class, groups are shuffled with a fixed-seed RNG and split 70 / 15 / 15 by group count.
- Val and Test contain **only clean originals** (no augmentations are produced by this notebook).
- The resulting file-index sets `train_idx / val_idx / test_idx` are pairwise disjoint.

In [ ]:
def get_group(path: str, label: int) -> str:
    """Group = '{class_id:02d}/{filename_stem}'.
    All files sharing the same stem (e.g. img.jpg and future img_aug001.jpg)
    map to the same group, preventing data leakage between splits.
    """
    return f"{int(label):02d}/{Path(path).stem}"

df["group"] = df.apply(lambda r: get_group(r["path"], r["label"]), axis=1)

group_class_counts = df.groupby("group")["label"].nunique()
assert group_class_counts.max() == 1, "Group spans multiple classes!"

rng = np.random.default_rng(SEED)
train_groups: set[str] = set()
val_groups:   set[str] = set()
test_groups:  set[str] = set()

for class_id in range(NUM_CLASSES):
    groups = np.array(sorted(df.loc[df["label"] == class_id, "group"].unique()), dtype=object)
    rng.shuffle(groups)
    n = len(groups)
    n_val  = round(n * VAL_RATIO)
    n_test = round(n * TEST_RATIO)
    n_train = n - n_val - n_test
    assert min(n_train, n_val, n_test) > 0, (
        f"Class {class_id}: train={n_train}, val={n_val}, test={n_test}"
    )
    train_groups.update(groups[:n_train])
    val_groups.update(groups[n_train:n_train + n_val])
    test_groups.update(groups[n_train + n_val:])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

train_idx = df.loc[df["group"].isin(train_groups), "index"].tolist()
val_idx   = df.loc[df["group"].isin(val_groups),   "index"].tolist()
test_idx  = df.loc[df["group"].isin(test_groups),  "index"].tolist()

split_df = pd.DataFrame([
    {"split": "Train", "groups": len(train_groups), "files": len(train_idx)},
    {"split": "Val",   "groups": len(val_groups),   "files": len(val_idx)},
    {"split": "Test",  "groups": len(test_groups),  "files": len(test_idx)},
])
display(split_df)

def split_of(g: str) -> str:
    if g in train_groups: return "Train"
    if g in val_groups:   return "Val"
    return "Test"

df["split"] = df["group"].map(split_of)
per_class_split = (
    df.groupby(["class_name", "split"]).size().unstack(fill_value=0).reindex(CLASS_NAMES)
)
per_class_split[["Train", "Val", "Test"]] = per_class_split[["Train", "Val", "Test"]].fillna(0).astype(int)
display(per_class_split)

assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)

## Phase 7 — Exploratory Data Analysis (EDA on Train split)

All EDA below is computed **only on the training split** so that no information from Val/Test ever leaks into model-selection decisions. We confirm:

- class balance,
- a visual sample of representative training images,
- aspect-ratio distribution (expecting all 1:1 since the dataset is already resized to 224×224),
- exact image dimensions for every train file.

In [ ]:
train_df = df[df["split"] == "Train"].copy()

fig, ax = plt.subplots(figsize=(10, 5))
counts = train_df["class_name"].value_counts().reindex(CLASS_NAMES)
sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="magma")
for i, v in enumerate(counts.values):
    ax.text(i, v + 0.5, str(v), ha="center", va="bottom", fontsize=9)
ax.set_title(f"Train split — per-class count (n={len(train_df)})")
ax.set_xlabel("class"); ax.set_ylabel("# images")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "train_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

rng_vis = np.random.default_rng(SEED)
picked_classes = rng_vis.choice(CLASS_NAMES, size=min(6, NUM_CLASSES), replace=False)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for ax, cls in zip(axes, picked_classes):
    pool = train_df[train_df["class_name"] == cls]["path"].tolist()
    p = pool[rng_vis.integers(0, len(pool))]
    img = Image.open(p).convert("RGB")
    ax.imshow(img); ax.set_title(cls, fontsize=10); ax.axis("off")
plt.suptitle("Sample training images (one per class)", fontsize=14)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "sample_train_images.png", dpi=120, bbox_inches="tight")
plt.show()

ar_values = []
dim_set = set()
for p in train_df["path"].sample(min(500, len(train_df)), random_state=SEED):
    with Image.open(p) as im:
        w, h = im.size
    ar_values.append(w / h)
    dim_set.add((w, h))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ar_values, bins=20, color="steelblue", edgecolor="black")
ax.set_title(f"Aspect-ratio distribution (train, sample={len(ar_values)})")
ax.set_xlabel("width / height"); ax.set_ylabel("count")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "aspect_ratio_train.png", dpi=120, bbox_inches="tight")
plt.show()

dim_summary = pd.DataFrame(
    sorted(dim_set), columns=["width", "height"]
).describe()[["width", "height"]]
display(dim_summary)
assert dim_set == {(IMAGE_SIZE, IMAGE_SIZE)}, (
    f"Expected all images to be {IMAGE_SIZE}x{IMAGE_SIZE}, found: {dim_set}"
)

## Phase 9 — Preprocessing & Augmentation Pipeline

Train transforms (data augmentation applied **only** to the training split):

- `RandomResizedCrop(224, scale=(0.8, 1.0))` — spatial jitter
- `RandomHorizontalFlip(p=0.5)`
- `RandomVerticalFlip(p=0.2)` — cosmetic bottles are visually symmetric
- `ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)`
- `RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))`
- `RandomGrayscale(p=0.05)`
- `ToTensor()`
- `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])` — ImageNet statistics for ResNet18 transfer learning
- `RandomErasing(p=0.1, scale=(0.02, 0.1))` — regularization

Val / Test transforms (deterministic, no augmentation):

- `Resize(int(224 * 1.15))` → `CenterCrop(224)`
- `ToTensor()` → ImageNet `Normalize`

No test-time augmentation (TTA) is applied in this preprocessing stage — TTA, if used, will be evaluated separately during the training stage.

In [ ]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

VAL_TEST_TRANSFORM = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.15)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = Subset(
    datasets.ImageFolder(DATASET_ROOT, transform=TRAIN_TRANSFORM), train_idx
)
val_dataset = Subset(
    datasets.ImageFolder(DATASET_ROOT, transform=VAL_TEST_TRANSFORM), val_idx
)
test_dataset = Subset(
    datasets.ImageFolder(DATASET_ROOT, transform=VAL_TEST_TRANSFORM), test_idx
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

sample_batch = next(iter(train_loader))
print(f"Train batch: images={tuple(sample_batch[0].shape)}, labels={tuple(sample_batch[1].shape)}")
print(f"Image range: [{sample_batch[0].min():.3f}, {sample_batch[0].max():.3f}]")

split_label_dist = pd.DataFrame({
    "class_name": CLASS_NAMES,
    "Train": np.bincount(
        df.loc[df["group"].isin(train_groups), "label"].values, minlength=NUM_CLASSES
    ),
    "Val": np.bincount(
        df.loc[df["group"].isin(val_groups), "label"].values, minlength=NUM_CLASSES
    ),
    "Test": np.bincount(
        df.loc[df["group"].isin(test_groups), "label"].values, minlength=NUM_CLASSES
    ),
}).set_index("class_name")
display(split_label_dist)

rng_vis = np.random.default_rng(SEED)
n_show = 15
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()
for i in range(n_show):
    ax = axes[i]
    idx_in_train = int(rng_vis.integers(0, len(train_dataset)))
    img, label = train_dataset[idx_in_train]
    img_np = img.permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_np = np.clip(img_np * std + mean, 0, 1)
    ax.imshow(img_np)
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample augmented training images (denormalized)", fontsize=14)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "augmentation_samples.png", dpi=120, bbox_inches="tight")
plt.show()

## Phase 11 — Preprocessing Summary & Export

### Dataset summary

| Item | Value |
|---|---|
| Source | `total_practice/practice_2_2/data_clean_224` |
| Total files | 2857 |
| Classes | 10 |
| Image size | 224 × 224 × 3 |
| MD5 verified | ✓ |
| Cross-class leaks | 0 |
| Intra-class duplicates | 0 |

### Split summary (group-aware, stratified)

| Split | Ratio | Groups | Files |
|---|---|---|---|
| Train | 70% | (see notebook output) | (see notebook output) |
| Val   | 15% | (see notebook output) | (see notebook output) |
| Test  | 15% | (see notebook output) | (see notebook output) |

### Transforms

- **Train**: RandomResizedCrop + HFlip + VFlip + ColorJitter + Affine + Grayscale + ToTensor + ImageNet Normalize + RandomErasing.
- **Val / Test**: Resize(257) + CenterCrop(224) + ToTensor + ImageNet Normalize.

### Leakage prevention notes

- Split is by **group** (`{class_id}/{filename_stem}`), not by individual file. Future augmentation files added under the same stem will automatically land in the same split.
- Val and Test contain only clean originals (no augmentation files exist in this dataset).
- Test indices are saved **before** training starts; nothing during training may read from `test_idx`.

### Next step

Load the exported `preprocessing_split.json` and use the same `TRAIN_TRANSFORM` / `VAL_TEST_TRANSFORM` definitions when training the ImageNet-pretrained ResNet18 backbone.

In [ ]:
split_config = {
    "seed": SEED,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "dataset_root": str(DATASET_ROOT),
    "class_names": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "total_files": int(total_files),
    "unique_hashes": int(unique_hashes),
    "cross_class_leaks": int(cross_class_leaks),
    "intra_dups": int(intra_dups),
    "train_indices": [int(i) for i in train_idx],
    "val_indices":   [int(i) for i in val_idx],
    "test_indices":  [int(i) for i in test_idx],
    "train_groups": sorted(train_groups),
    "val_groups":   sorted(val_groups),
    "test_groups":  sorted(test_groups),
    "transforms": {
        "train":    str(TRAIN_TRANSFORM),
        "val_test": str(VAL_TEST_TRANSFORM),
    },
    "imagenet_stats": {
        "mean": [0.485, 0.456, 0.406],
        "std":  [0.229, 0.224, 0.225],
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = OUTPUT_DIR / "preprocessing_split.json"
SPLIT_PATH.write_text(json.dumps(split_config, indent=2))
print(f"Split config saved to: {SPLIT_PATH}")

summary_df = pd.DataFrame([
    ["Dataset",         DATASET_ROOT.name],
    ["Total files",      total_files],
    ["Classes",          NUM_CLASSES],
    ["Train / Val / Test",
     f"{len(train_idx)} / {len(val_idx)} / {len(test_idx)}"],
    ["Train groups",     len(train_groups)],
    ["Val groups",       len(val_groups)],
    ["Test groups",      len(test_groups)],
    ["Cross-class leaks", cross_class_leaks],
    ["MD5 verified",     "Yes"],
    ["Split config",     str(SPLIT_PATH.relative_to(PROJECT_ROOT))],
], columns=["key", "value"])
display(summary_df)